# 오류와 읽기 도구·쓰기 도구 분리

오류 코드와 제공 경로를 분리합니다.

## 준비

## 코드를 읽기 전에

`read_order()`는 예상 가능한 ‘주문 없음’을 예외 대신 `ok=False`와 오류 코드가 든 사전으로 반환합니다. 호출자는 성공 결과와 업무상 미발견을 같은 반환 형식에서 구분할 수 있습니다. 반면 코드 결함이나 연결 실패까지 모두 같은 오류 코드로 바꾸면 원인을 잃으므로 예상 범위를 정해야 합니다.

쓰기 도구의 인자에는 `confirmed`가 없습니다. 사용자의 확인된 주문 ID는 애플리케이션 상태인 `confirmed_order_ids`에 저장되고, 도구 실행 시 요청 ID가 그 집합에 있는지 다시 검사합니다. 모델이 만든 문자열은 사용자 승인 증거가 될 수 없습니다.

In [1]:
# 상위 폴더에서 환경 파일과 공개 주문 CSV를 찾습니다.
from pathlib import Path
from dotenv import load_dotenv

def find_up(name: str) -> Path | None:
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        candidate = folder / name
        if candidate.exists():
            return candidate
    return None

load_dotenv(find_up(".env"))
data_path = find_up("data/orders.csv")
assert data_path is not None, "공개 묶음의 챕터 폴더 또는 notebooks 폴더에서 실행합니다."
print("Python 실습 준비 완료")


Python 실습 준비 완료


### 다음 셀을 읽는 방법

다음 셀의 `ORDERS`는 외부 시스템 대신 사용할 교육용 주문 목록입니다. 각 사전은 주문 ID·상태·금액을 담고, 뒤의 읽기 도구는 이 목록에서 한 행을 찾습니다. 목록을 선언하는 것만으로 읽기나 취소 동작이 실행되지는 않습니다.


In [2]:
# 읽기 도구가 조회할 주문 행을 메모리에 준비합니다.
from langchain.tools import tool
import pandas as pd
ORDERS = pd.read_csv(data_path).to_dict(orient="records")


## 읽기 도구

In [3]:
# 없는 주문은 빈 성공 결과와 구분해 반환합니다.
@tool
def read_order(order_id: str) -> dict:
    """주문 ID로 조회하며 데이터를 변경하지 않습니다."""
    row = next((r for r in ORDERS if r["order_id"] == order_id), None)
    return {"ok": True, "order": row} if row else {"ok": False, "error": "NOT_FOUND", "order_id": order_id}
print(read_order.invoke({"order_id": "O999"}))


{'ok': False, 'error': 'NOT_FOUND', 'order_id': 'O999'}


## 쓰기 도구

In [4]:
# 취소 요청 도구는 실행 직전 앱의 확인 상태를 검사합니다.
@tool
def request_order_cancellation(order_id: str) -> dict:
    """애플리케이션이 확인한 주문의 취소 요청을 기록합니다."""
    if not app_user_confirmed or order_id not in confirmed_order_ids:
        raise PermissionError("앱에서 사용자 확인을 받지 않았습니다.")
    return {"ok": True, "status": "cancellation_requested", "order_id": order_id}


## 확인 전

In [5]:
# 도구를 숨긴 상태에서도 직접 호출이 거부되는지 확인합니다.
app_user_confirmed = False  # 신뢰하는 애플리케이션 상태의 실습값
confirmed_order_ids: set[str] = set()
available_tools = [read_order]
print({"confirmed": app_user_confirmed, "tools": [t.name for t in available_tools]})
assert request_order_cancellation not in available_tools
try:
    request_order_cancellation.invoke({"order_id": "O101"})
except PermissionError as error:
    print(type(error).__name__, str(error))
else:
    raise AssertionError("확인 전 요청이 허용되었습니다.")


{'confirmed': False, 'tools': ['read_order']}
PermissionError 앱에서 사용자 확인을 받지 않았습니다.


## 애플리케이션이 사용자 확인을 기록한 뒤

In [6]:
# 확인된 앱 상태에서만 취소 요청 결과를 만듭니다.
app_user_confirmed = True  # 실제 서비스에서는 UI 확인 결과를 서버가 기록합니다.
confirmed_order_ids = {"O101"}  # 확인한 작업의 주문 ID까지 제한합니다.
available_tools = [read_order, request_order_cancellation] if app_user_confirmed else [read_order]
print({"confirmed": app_user_confirmed, "tools": [t.name for t in available_tools]})
result = request_order_cancellation.invoke({"order_id": "O101"})
print(result)
assert result["status"] == "cancellation_requested" and result["order_id"] == "O101"

try:
    request_order_cancellation.invoke({"order_id": "O103"})
except PermissionError as error:
    print("다른 주문 거부:", type(error).__name__)
else:
    raise AssertionError("확인하지 않은 주문이 허용되었습니다.")


{'confirmed': True, 'tools': ['read_order', 'request_order_cancellation']}
{'ok': True, 'status': 'cancellation_requested', 'order_id': 'O101'}
다른 주문 거부: PermissionError


## 다른 상황에 적용하기

메일 검색과 메일 발송을 한 도구 목록에 함께 제공하면 모델의 선택 오류가 곧 외부 부작용으로 이어질 수 있습니다. 검색 단계에는 읽기 도구만 제공하고, 수신자·본문을 사람이 확인한 뒤 해당 초안 ID에 한해 발송 도구를 열 수 있습니다. 도구 공개 여부와 함수 내부의 권한 재검사를 함께 두어 한 경계가 빠져도 다른 경계가 남게 합니다.